# Find selection coefficient against individual missense variants

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import scipy as sc
from scipy.integrate import quad
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde
from scipy.stats import gmean, linregress, norm, beta, uniform, lognorm, pearsonr, spearmanr
from scipy.integrate import simpson, cumulative_trapezoid, trapezoid
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.integrate import simpson
from scipy import stats
import glob
import os
import math
import re
import csv
from tqdm import tqdm
import time
import joblib
from joblib import Parallel, delayed, parallel_backend
import multiprocessing
import pickle
import random
import ast
import json
import Generate_missense

In [2]:
s_values = -np.logspace(-6, 2, num=100)
log_s_grid = np.log10(-s_values)
dx = log_s_grid[1] - log_s_grid[0]

### Get AlphaMissense based scores

In [3]:
score_col = "AlphaMissense"
input_file = "../Data/missense/df_missense_AC.txt.gz"
output_file = f"/n/data2/hms/dbmi/sunyaev/lab/pkar/Missense_analysis/final/{score_col}.pkl"
lof_ref_path = "../LoF_selection/LoF_s_het.txt.gz"

# MLE params
init_c = 0.1
init_beta = 1
init_log_sigma = np.log(0.5551)
bounds = [(-4, 4), (0, 10), (np.log(dx), np.log(10.0))]

_,_,_ = Generate_missense.run_pipeline(input_file, output_file, score_col, allele_count_max=5000,
    sfs_mle_pkl="/n/data2/hms/dbmi/sunyaev/lab/pkar/demography_SFS/SFS_all_missense.pkl",
    sfs_variant_pkl="/n/data2/hms/dbmi/sunyaev/lab/pkar/demography_SFS/SFS_all.pkl",
    # MLE params
    init_c=init_c,
    init_beta=init_beta,
    init_log_sigma=init_log_sigma,
    bounds=bounds,
    maxiter=400,
    fatol=1e-8,
    # parallel
    n_jobs=-1,
    backend="multiprocessing",
    verbose=10,
    # outputs
    gene_mle_out_dir=None,
    # optional LoF dedup
    lof_ref_path=lof_ref_path,
    lof_ref_score_col="Posterior_CI10_lower",
    key_cols=None)

[Parallel(n_jobs=-1)]: Using backend MultiprocessingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    8.3s
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:    8.3s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:    8.5s
[Parallel(n_jobs=-1)]: Done  58 tasks      | elapsed:    8.6s
[Parallel(n_jobs=-1)]: Done  73 tasks      | elapsed:    8.7s
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:    8.9s
[Parallel(n_jobs=-1)]: Done 105 tasks      | elapsed:    9.1s
[Parallel(n_jobs=-1)]: Done 122 tasks      | elapsed:    9.4s
[Parallel(n_jobs=-1)]: Done 141 tasks      | elapsed:    9.7s
[Parallel(n_jobs=-1)]: Done 160 tasks      | elapsed:   10.0s
[Parallel(n_jobs=-1)]: Done 181 tasks      | elapsed:   10.3s
[Parallel(n_jobs=-1)]: Done 202 tasks      | elapsed:   10.7s
[Parallel(n_jobs=-1)]: Done 225 tasks      |

AttributeError: Can't get local object 'attach_medians_all_genes.<locals>._one_gene'

### Get popEVE based scores

In [3]:
score_col = "popEVE_neg"
input_file = "../Data/missense/df_missense_AC.txt.gz"
output_file = f"/n/data2/hms/dbmi/sunyaev/lab/pkar/Missense_analysis/final/{score_col}.pkl"
lof_ref_path = "../LoF_selection/LoF_s_het.txt.gz"

# MLE params
init_c = 4
init_beta = 1
init_log_sigma = np.log(0.5551)
bounds = [(0, 8), (0, 10), (np.log(dx), np.log(10.0))]

_,_,_ = Generate_missense.run_pipeline(input_file, output_file, score_col, allele_count_max=5000,
    sfs_mle_pkl="/n/data2/hms/dbmi/sunyaev/lab/pkar/demography_SFS/SFS_all_missense.pkl",
    sfs_variant_pkl="/n/data2/hms/dbmi/sunyaev/lab/pkar/demography_SFS/SFS_all.pkl",
    # MLE params
    init_c=init_c,
    init_beta=init_beta,
    init_log_sigma=init_log_sigma,
    bounds=bounds,
    maxiter=400,
    fatol=1e-8,
    # parallel
    n_jobs=-1,
    backend="multiprocessing",
    verbose=10,
    # outputs
    gene_mle_out_dir=None,
    # optional LoF dedup
    lof_ref_path=lof_ref_path,
    lof_ref_score_col="Posterior_CI10_lower",
    key_cols=None)

/home/prk534/Pop_gen_simulator/Paper SFS/Missense/Generate_missense.py:397: DtypeWarning: Columns (13,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_orig = pd.read_csv(input_file, sep="\t", compression="gzip")
[Parallel(n_jobs=-1)]: Using backend MultiprocessingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    6.5s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    6.5s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    6.8s
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:    6.9s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:    7.2s
[Parallel(n_jobs=-1)]: Done  58 tasks      | elapsed:    7.5s
[Parallel(n_jobs=-1)]: Done  73 tasks      | elapsed:    7.9s
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done 105 tasks      | elapsed:    8.8s
[Parallel(n_jobs=-1)]: Done 122 tasks      | elapsed:    9.3s
[Parallel(n_jobs=-1)]: Done 141 tasks      | e

Duplicate resolution summary: {'all_nan': 0, 'some_valid': 0, 'all_valid': 7875}
Saved: /n/data2/hms/dbmi/sunyaev/lab/pkar/Missense_analysis/final/popEVE_neg.pkl
Columns added: s_prior_popEVE_neg, s_popEVE_neg
Time: 10967.83 s


### Get ESM_1b based scores

In [ ]:
score_col = "esm_score_neg"
input_file = "../Data/missense/df_missense_AC.txt.gz"
output_file = f"/n/data2/hms/dbmi/sunyaev/lab/pkar/Missense_analysis/final/{score_col}.pkl"
lof_ref_path = "../LoF_selection/LoF_s_het.txt.gz"

# MLE params
init_c = 5
init_beta = 1
init_log_sigma = np.log(0.5551)
bounds = [(-5, 25), (0, 15), (np.log(dx), np.log(10.0))]

_,_,_ = Generate_missense.run_pipeline(input_file, output_file, score_col, allele_count_max=5000,
    sfs_mle_pkl="/n/data2/hms/dbmi/sunyaev/lab/pkar/demography_SFS/SFS_all_missense.pkl",
    sfs_variant_pkl="/n/data2/hms/dbmi/sunyaev/lab/pkar/demography_SFS/SFS_all.pkl",
    # MLE params
    init_c=init_c,
    init_beta=init_beta,
    init_log_sigma=init_log_sigma,
    bounds=bounds,
    maxiter=400,
    fatol=1e-8,
    # parallel
    n_jobs=-1,
    backend="multiprocessing",
    verbose=10,
    # outputs
    gene_mle_out_dir=None,
    # optional LoF dedup
    lof_ref_path=lof_ref_path,
    lof_ref_score_col="Posterior_CI10_lower",
    key_cols=None)

/home/prk534/Pop_gen_simulator/Paper SFS/Missense/Generate_missense.py:397: DtypeWarning: Columns (13,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_orig = pd.read_csv(input_file, sep="\t", compression="gzip")
[Parallel(n_jobs=-1)]: Using backend MultiprocessingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:   11.7s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   11.9s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:   12.1s
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:   12.3s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:   12.5s
[Parallel(n_jobs=-1)]: Done  58 tasks      | elapsed:   12.9s
[Parallel(n_jobs=-1)]: Done  73 tasks      | elapsed:   13.3s
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:   13.8s
[Parallel(n_jobs=-1)]: Done 105 tasks      | elapsed:   14.3s
[Parallel(n_jobs=-1)]: Done 122 tasks      | elapsed:   14.9s
[Parallel(n_jobs=-1)]: Done 141 tasks      | e

Duplicate resolution summary: {'all_nan': 0, 'some_valid': 0, 'all_valid': 9109}


# Coalesce these scores into one file

In [2]:
def outer_merge_coalesce(df_left, df_right, key_cols):
    """
    Outer merge df_left and df_right on key_cols.
    For overlapping non-key columns, coalesce values (prefer left, fill from right),
    and drop the _x/_y duplicates.
    """
    overlap = [c for c in df_left.columns if c in df_right.columns and c not in key_cols]

    merged = df_left.merge(
        df_right,
        on=key_cols,
        how="outer",
        suffixes=("", "_r"),
        copy=False,
        sort=False,
    )

    # coalesce overlapping columns
    for c in overlap:
        merged[c] = merged[c].combine_first(merged[f"{c}_r"])
        merged.drop(columns=[f"{c}_r"], inplace=True)

    return merged

In [ ]:
with open(f"/n/data2/hms/dbmi/sunyaev/lab/pkar/Missense_analysis/final/AlphaMissense.pkl", "rb") as f:
    df_AM = pickle.load(f)
with open(f"/n/data2/hms/dbmi/sunyaev/lab/pkar/Missense_analysis/final/popEVE_neg.pkl", "rb") as f:
    df_popeve = pickle.load(f)
with open(f"/n/data2/hms/dbmi/sunyaev/lab/pkar/Missense_analysis/final/esm_score_neg.pkl", "rb") as f:
    df_esm1b = pickle.load(f)

print("Done")

key_cols = ["#CHROM", "POS", "REF", "ALT"]
df_all = outer_merge_coalesce(df_AM, df_popeve, key_cols)
df_all = outer_merge_coalesce(df_all, df_esm1b, key_cols)


In [4]:
print(len(df_all))
print(df_all[df_all.duplicated(subset=['#CHROM', 'POS', 'REF', 'ALT'], keep=False)])

60256263
Empty DataFrame
Columns: [gene_id, proteinmpnn_llr_neg, rasp_score, polyphen_score, cpt1_score, popEVE_neg, ESM_1v_neg, cadd_score, gpn_msa_score, #CHROM, POS, REF, ALT, QUAL, MR, gene_symbol, AlphaMissense, MPC, esm_score_neg, allele_count, allele_number, MisFit_D, MisFit_S, PrimateAI-3D, REVEL, MR_rank, AlphaMissense_norm, s_prior_AlphaMissense_norm, s_AlphaMissense_norm, s_prior_popEVE_neg, s_popEVE_neg, s_prior_esm_score_neg, s_esm_score_neg]
Index: []

[0 rows x 33 columns]


In [7]:
print((df_AM["AlphaMissense"].notna()).sum())
print((df_all["AlphaMissense"].notna()).sum())

59182427
59182427


In [9]:
# with open("/n/data2/hms/dbmi/sunyaev/lab/pkar/Missense_analysis/final/all_new_scores.pkl", "wb") as f:
#     pickle.dump(df_all, f)

In [2]:
with open(f"/n/data2/hms/dbmi/sunyaev/lab/pkar/Missense_analysis/final/all_new_scores.pkl", "rb") as f:
    df_all_variants = pickle.load(f)
print("Done")

Done


In [3]:
len(df_all_variants)

60256263

In [5]:
df_gene_names = pd.read_csv("../Data/ENS2Gene.txt", sep="\t", header=None, names=["gene_id", "gene_name"])
df_all_variants_full = df_all_variants.merge(df_gene_names, on = "gene_id", how = "left")

In [6]:
df_all_variants_full[['gene_id','#CHROM', 'POS', 'REF', 'ALT', 'QUAL', 'MR', 'gene_name', 'allele_count', 'allele_number', 's_prior_AlphaMissense_norm',
                      's_AlphaMissense_norm', 's_prior_popEVE_neg', 's_popEVE_neg', 's_prior_esm_score_neg', 's_esm_score_neg']].to_csv("/n/data2/hms/dbmi/sunyaev/lab/pkar/Missense_analysis/final/Supplementary_Table_6.txt.gz", sep="\t", index=False, compression="gzip")